In [1]:
import polars as pl
import mappy as mp
import os
from collections import Counter
from collections import OrderedDict

In [2]:
input ='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/0_data/EMP/ERR4994172.fastq.gz'
output_align='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align'
output_meta='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta'

In [3]:
#setting class so it can be used
#dataclass
class AmpliconDataset:
    input_file: str = None
    basename: str = None
    output_directory: str = None

In [4]:
# Initialize counters for primary, secondary, supplementary, and duplicates
alignment_counts = Counter({
    'primary': 0,
    'secondary': 0,
    'supplementary': 0,
    'duplicates': 0,
    'total': 0
})

In [17]:
#check alignment_counts
print(alignment_counts)

Counter({'primary': 0, 'secondary': 0, 'supplementary': 0, 'duplicates': 0, 'total': 0})


In [3]:
index='/home/n11702427/m/users/r_nurdiansyah/wagtail/0_database/danica.mmi'

In [4]:
#read index to mappy
ref = mp.Aligner(index, preset = 'sr')
if not ref: raise Exception("ERROR: failed to load/build index")
#print to console if index is loaded from the file

In [38]:
#testing mappy to read fastq
for name, seq, qual in mp.fastx_read(input, read_comment=False):
    #print("{}\t{}\t{}".format(name, seq, qual))
    r1_name = name
    print(r1_name)

ERR4994172.1
ERR4994172.2
ERR4994172.3
ERR4994172.4
ERR4994172.5
ERR4994172.6
ERR4994172.7
ERR4994172.8
ERR4994172.9
ERR4994172.10
ERR4994172.11
ERR4994172.12
ERR4994172.13
ERR4994172.14
ERR4994172.15
ERR4994172.16
ERR4994172.17
ERR4994172.18
ERR4994172.19
ERR4994172.20
ERR4994172.21
ERR4994172.22
ERR4994172.23
ERR4994172.24
ERR4994172.25
ERR4994172.26
ERR4994172.27
ERR4994172.28
ERR4994172.29
ERR4994172.30
ERR4994172.31
ERR4994172.32
ERR4994172.33
ERR4994172.34
ERR4994172.35
ERR4994172.36
ERR4994172.37
ERR4994172.38
ERR4994172.39
ERR4994172.40
ERR4994172.41
ERR4994172.42
ERR4994172.43
ERR4994172.44
ERR4994172.45
ERR4994172.46
ERR4994172.47
ERR4994172.48
ERR4994172.49
ERR4994172.50
ERR4994172.51
ERR4994172.52
ERR4994172.53
ERR4994172.54
ERR4994172.55
ERR4994172.56
ERR4994172.57
ERR4994172.58
ERR4994172.59
ERR4994172.60
ERR4994172.61
ERR4994172.62
ERR4994172.63
ERR4994172.64
ERR4994172.65
ERR4994172.66
ERR4994172.67
ERR4994172.68
ERR4994172.69
ERR4994172.70
ERR4994172.71
ERR4994172.72
E

In [5]:
# Initialize counters -> for metadata
total_mapq = 0
alignment_counter = Counter()

In [6]:
result_per_i = {}
for name, seq, qual in mp.fastx_read(input, read_comment=False):
    #alignment_counts['total'] += 1
    is_mapped = False #to track duplicates
    #for hits in ref.map(seq, cs=True, MD=True):
    for hits in ref.map(seq):
        if hits.is_primary:
            result_per_i[name] = (hits.ctg)
            alignment_counter['mapped'] += 1
            total_mapq += hits.mapq
            is_mapped = True
            break # Only consider primary alignment
        else:
            alignment_counter['unmapped'] += 1
    #count the number of hits using Counter
ctg_counts = Counter(result_per_i.values())    
# print(result_per_i)

In [7]:
# Calculate mapping percentage
total_reads = alignment_counter["mapped"] + alignment_counter["unmapped"]
mapping_percentage = (alignment_counter["mapped"] / total_reads) * 100 if total_reads > 0 else 0
average_mapq = total_mapq / alignment_counter["mapped"] if alignment_counter["mapped"] > 0 else 0

# Print the results
print(f"Total Reads: {total_reads}")
print(f"Mapped Reads: {alignment_counter['mapped']}")
print(f"Unmapped Reads: {alignment_counter['unmapped']}")
print(f"Mapping Percentage: {mapping_percentage:.2f}%")
print(f"Average MAPQ: {average_mapq:.2f}")

Total Reads: 11136
Mapped Reads: 11136
Unmapped Reads: 0
Mapping Percentage: 100.00%
Average MAPQ: 2.94


In [8]:
#prepare metadata into dataframe with column name: sample-id, total reads, mapped reads, unmapped reads, mapping percentage, average mapq
metadata = pl.DataFrame({
    'sample-id': ['ERR4994172'],
    'total reads': [total_reads],
    'mapped reads': [alignment_counter['mapped']],
    'unmapped reads': [alignment_counter['unmapped']],
    'mapping percentage': [mapping_percentage],
    'average mapq': [average_mapq]
})

In [9]:
print(metadata)

shape: (1, 6)
┌────────────┬─────────────┬──────────────┬────────────────┬────────────────────┬──────────────┐
│ sample-id  ┆ total reads ┆ mapped reads ┆ unmapped reads ┆ mapping percentage ┆ average mapq │
│ ---        ┆ ---         ┆ ---          ┆ ---            ┆ ---                ┆ ---          │
│ str        ┆ i64         ┆ i64          ┆ i64            ┆ f64                ┆ f64          │
╞════════════╪═════════════╪══════════════╪════════════════╪════════════════════╪══════════════╡
│ ERR4994172 ┆ 11136       ┆ 11136        ┆ 0              ┆ 100.0              ┆ 2.936692     │
└────────────┴─────────────┴──────────────┴────────────────┴────────────────────┴──────────────┘


In [42]:
# Prepare metadata dictionary
metadata = {
    "Total Reads": total_reads,
    "Mapped Reads": alignment_counter["mapped"],
    "Unmapped Reads": alignment_counter["unmapped"],
    "Mapping Percentage": f"{mapping_percentage:.2f}%",
    "Average Mapping Quality": f"{average_mapq:.2f}"
}

In [50]:
filename=(f"/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta/{basename}_metadata.txt")
with open(filename, 'w') as f:
    for key, value in metadata.items():
        f.write(f"{key}: {value}\n")
    print(f"Metadata successfully saved to {filename}")

Metadata successfully saved to /home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta/ERR4994172_metadata.txt


In [44]:
print(alignment_counter["mapped"])

11136


In [45]:
print(ctg_counts)

Counter({'FLASV957672.1365;tax=d:Bacteria,p:Bacteroidota,c:Bacteroidia,o:Bacteroidales,f:Muribaculaceae,g:MFD_g_957672,s:MFD_s_957672;': 2875, 'FLASV60.1391;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Lactobacillaceae,g:Lactobacillus,s:MFD_s_60;': 2148, 'FLASV89.1392;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Lactobacillaceae,g:Lactobacillus,s:MFD_s_89;': 871, 'FLASV927273.1364;tax=d:Bacteria,p:Bacteroidota,c:Bacteroidia,o:Bacteroidales,f:Muribaculaceae,g:MFD_g_802620,s:MFD_s_927273;': 592, 'FLASV344784.1362;tax=d:Bacteria,p:Bacteroidota,c:Bacteroidia,o:Bacteroidales,f:Muribaculaceae,g:MFD_g_344784,s:MFD_s_344784;': 343, 'FLASV903987.1369;tax=d:Bacteria,p:Bacteroidota,c:Bacteroidia,o:Bacteroidales,f:Muribaculaceae,g:MFD_g_812857,s:MFD_s_903987;': 305, 'FLASV972609.1365;tax=d:Bacteria,p:Bacteroidota,c:Bacteroidia,o:Bacteroidales,f:Muribaculaceae,g:MFD_g_849107,s:MFD_s_972609;': 237, 'FLASV1877.1392;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:La

In [46]:
#get the base name of the input file
basename = os.path.basename(input).split('.')[0]
#remove the extension of the file
#basename = os.path.splitext(basename).split('.')[0]
print(basename)

ERR4994172


In [47]:
#record the ctg_counts dictionary to a tsv file using polars
df1 = pl.from_dict(ctg_counts)
df1 = df1.transpose(include_header=True)
#add sample column on the first column and fill the value with user's input (sample name)
df1.columns = ['contig', 'count']
df1 = df1.with_columns(pl.Series("sample", [basename]*len(df1)))
#reorder the column to match the biobox script input while delete the unnecessary columns
df1 = df1[["sample", "contig", "count"]]
#sort based on count
df2 = df1.sort("count", descending=True)
df2[:4]

sample,contig,count
str,str,i64
"""ERR4994172""","""FLASV957672.13…",2875
"""ERR4994172""","""FLASV60.1391;t…",2148
"""ERR4994172""","""FLASV89.1392;t…",871
"""ERR4994172""","""FLASV927273.13…",592


In [49]:
df2.write_csv(f"/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align/{basename}_alignment.tsv", separator='\t')

In [18]:
for i in input_files:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [6]:
for i in input:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [9]:
# Check if input is a file or a directory and list all files in the directory
input_files=[]
printout=[]
#d = AmpliconDataset()
if isinstance (input, str):
    input = [input] #convert to list if input is a string
    
input_file = sorted(input_files)
file_names = [os.path.basename(f) for f in input_files]

d = AmpliconDataset()
datasets = []
for file in file_names:
    base = file.split('.')[0]
    d.basename = base
    datasets.append(d)

for input in input_file:
    d.input_file = input
    datasets.append(d)

print(d.input_file)
print(d.basename)

None
None


In [63]:
#making directory, no need to run this
for d in datasets:
    file = d.basename
    # Extract parent directory name (first 3 digits after "ERR")
    if len(file) < 6:
        logging.error(f"Filename {file} is too short")
        sys.exit(1)
    parent_dir = file[:6]
    
    # Extract the last digit to determine the subdirectory ("000" to "009")
    sub_dir_num = int(file[-3:]) % 1000
    
    # Create the full path for the accession directory
    new_dir = os.path.join(output, f"{parent_dir}", f"{sub_dir_num}", file)
    d.output_directory = new_dir
          
    #Create new directory if not exist
    if not os.path.exists(new_dir):
        os.makedirs(new_dir, exist_ok=True)
        logging.info(f"Created directory: {new_dir}")
    else:
        logging.info(f"Directory already exists: {new_dir}")

In [12]:
# Create empty lists to store alignment data and metadata counts
alignments = []
has_primary_flags = {}

In [20]:
for i in input:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [18]:
# Process each input sequence in the input file (assuming FASTA/FASTQ format)
for i in input:
    result_per_i = {}
    for name, seq, qual in mp.fastx_read(i):
        alignment_counts['total'] += 1
        #has_primary = False  # To track duplicates

        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
        #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)
        # # Perform alignment and store results
        # for hit in aligner.map(seq):
        #     # Store alignment result in list
        #     alignments.append([name, hit.ctg])

        #     if hit.is_primary and not has_primary:
        #         alignment_counts['primary'] += 1
        #         has_primary = True
        #     elif hit.is_secondary:
        #         alignment_counts['secondary'] += 1
        #     if hit.is_supplementary:
        #         alignment_counts['supplementary'] += 1
        #     if has_primary and hit.is_primary:  # Mark duplicates
        #         alignment_counts['duplicates'] += 1

print(alignments)
    # Convert alignment list to Polars DataFrame and save as TSV
    # alignment_df = pl.DataFrame(alignments, schema=["read_name", "ctg"])
    # alignment_df.write_csv(alignment_output, separator="\t", has_header=True)

Counter()
[]


In [53]:
for da in datasets:
    i = da.input_file
    d = da.output_directory

    print("Running analysis for {}".format(i))
    result_per_i = {}
    ctg_counts = Counter()
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
        #count the number of contigs using Counter
        ctg_counts.update(result_per_i.values())
        print(ctg.counts)

Running analysis for V3V4005.fastq.gz
Running analysis for V1V2002.fastq.gz
Running analysis for V1V2001.fastq.gz
Running analysis for V4V6007.fastq.gz
Running analysis for V4V6008.fastq.gz
Running analysis for V1V3004.fastq.gz
Running analysis for V3V4006.fastq.gz
Running analysis for V1V3003.fastq.gz
